<a href="https://colab.research.google.com/github/evangeliachrysostomou2-create/Eva/blob/main/Evangelia_10_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import sys
print(sys.version)

3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


In [25]:
# Import the libraries required for the analysis.
import numpy as np
import pandas as pd
import scanpy as sc
import scipy
import statsmodels

# Print library versions to check the current environment.
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scanpy:", sc.__version__)
print("SciPy:", scipy.__version__)
print("Statsmodels:", statsmodels.__version__)

NumPy: 2.1.3
Pandas: 2.2.3
Scanpy: 1.10.4
SciPy: 1.16.3
Statsmodels: 0.15.0


In [26]:
# Import the ABC Atlas project cache module.
import abc_atlas_access.abc_atlas_cache.abc_project_cache as abc_cache

# Confirm that the ABC Atlas package is available.
print("ABC Atlas package imported successfully.")

ABC Atlas package imported successfully.


In [27]:
# Import the Google Colab Drive integration.
from google.colab import drive

# Mount Google Drive to access files stored in your account.
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
# Define the directory containing the ABC Atlas cached data.
CACHE_DIR = "/content/abc_atlas_data"

# Check whether the cache directory exists.
import os
print("Cache directory exists:", os.path.isdir(CACHE_DIR))

# List the contents of the directory if it exists.
if os.path.isdir(CACHE_DIR):
    print(os.listdir(CACHE_DIR)[:10])

Cache directory exists: True
['metadata', '_downloaded_data.json', '_manifest_last_used.txt', 'expression_matrices', 'releases']


In [29]:
# Initialize the ABC Atlas cache using the existing data directory.
cache = abc_cache.AbcProjectCache.from_cache_dir(CACHE_DIR)

# Confirm that the cache was initialized successfully.
print("ABC Atlas cache initialized successfully.")

ABC Atlas cache initialized successfully.


In [30]:
# Define the dataset name.
DATASET = "Zeng-Aging-Mouse-10Xv3"

# Load cell metadata from the ABC Atlas cache.
cell_metadata = cache.get_metadata_dataframe(
    directory=DATASET,
    file_name="cell_metadata"
)

# Display the number of cells and metadata columns.
print("Cell metadata shape:", cell_metadata.shape)

Cell metadata shape: (1162565, 28)


In [31]:
# Count unique donors in each age group.
donor_counts_by_age = (
    cell_metadata.groupby("donor_age")["donor_label"]
    .nunique()
    .sort_index()
)

# Display the number of donors for each age group.
print(donor_counts_by_age)

# Report the total number of unique donors.
print("\nTotal unique donors:", cell_metadata["donor_label"].nunique())

donor_age
18M      4
9 wks    3
P53      2
P54      4
P540     4
P541     4
P542     5
P543     5
P544     4
P545     3
P546     3
P547     2
P549     5
P55      2
P550     2
P551     1
P553     2
P56      9
P57      1
P58      4
P59      5
P60      3
P61      4
P62      4
P64      3
P65      2
P66      2
P67      5
P68      4
P69      7
Name: donor_label, dtype: int64

Total unique donors: 108


In [32]:
# Extract unique donor-age combinations and sort them.
donor_info = (
    cell_metadata[["donor_age", "donor_label"]]
    .drop_duplicates()
    .sort_values(["donor_age", "donor_label"])
    .reset_index(drop=True)
)

# Display the donor table dimensions.
print("Donor table shape:", donor_info.shape)

# Display the first five donors.
display(donor_info.head())

Donor table shape: (108, 2)


,donor_age,donor_label
0,18M,C57BL/6J-592557
1,18M,C57BL/6J-592558
2,18M,Snap25-IRES2-Cre;Ai14-463229
3,18M,Snap25-IRES2-Cre;Ai14-464753
4,9 wks,Snap25-IRES2-Cre;Ai14-473981


In [33]:
# Locate the raw gene expression matrix in the ABC Atlas dataset.
raw_path = cache.get_file_path(
    directory=DATASET,
    file_name=f"{DATASET}/raw",
)

# Open the expression matrix in read-only backed mode.
adata = sc.read_h5ad(raw_path, backed="r")

# Display the number of cells and genes.
print("Expression matrix shape:", adata.shape)

Expression matrix shape: (1162565, 32285)


In [34]:
# Load gene metadata from the ABC Atlas.
gene_metadata = cache.get_metadata_dataframe(
    directory="WMB-10X",
    file_name="gene"
)

# Display the dimensions and available columns.
print("Gene metadata shape:", gene_metadata.shape)
print("Columns:", gene_metadata.columns.tolist())

# Display the first five gene records.
display(gene_metadata.head())

Gene metadata shape: (32285, 5)
Columns: ['gene_identifier', 'gene_symbol', 'name', 'mapped_ncbi_identifier', 'comment']


,gene_identifier,gene_symbol,name,mapped_ncbi_identifier,comment
0,ENSMUSG00000051951,Xkr4,X-linked Kx blood group related 4,NCBIGene:497097,NaN
1,ENSMUSG00000089699,Gm1992,predicted gene 1992,NaN,NaN
2,ENSMUSG00000102331,Gm19938,"predicted gene, 19938",NaN,NaN
3,ENSMUSG00000102343,Gm37381,"predicted gene, 37381",NaN,NaN
4,ENSMUSG00000025900,Rp1,retinitis pigmentosa 1 (human),NCBIGene:19888,NaN


In [35]:
# Extract gene identifiers from the expression matrix.
expression_gene_ids = pd.Index(adata.var_names.astype(str))

# Extract gene identifiers from the gene metadata table.
metadata_gene_ids = pd.Index(
    gene_metadata["gene_identifier"].astype(str)
)

# Count identifiers that appear in both datasets.
common_ids = expression_gene_ids.intersection(metadata_gene_ids)

# Report identifier counts and mismatches.
print("Genes in expression matrix:", len(expression_gene_ids))
print("Genes in metadata:", len(metadata_gene_ids))
print("Matching gene identifiers:", len(common_ids))
print("Expression IDs missing from metadata:",
      len(expression_gene_ids.difference(metadata_gene_ids)))

Genes in expression matrix: 32285
Genes in metadata: 32285
Matching gene identifiers: 32207
Expression IDs missing from metadata: 78


In [36]:
# Identify gene IDs present in the expression matrix but absent from metadata.
missing_from_metadata = expression_gene_ids.difference(metadata_gene_ids)

# Display the first 10 unmatched expression identifiers.
print("First 10 unmatched expression IDs:")
for gene_id in missing_from_metadata[:10]:
    print(repr(gene_id))

First 10 unmatched expression IDs:
'gene_symbol\n4930594M22Rik    ENSMUSG00000075463\n4930594M22Rik    ENSMUSG00000085423\nName: gene_identifier, dtype: object'
'gene_symbol\n4930594M22Rik    ENSMUSG00000085423\nName: gene_identifier, dtype: object'
'gene_symbol\nAldoa    ENSMUSG00000030695\nAldoa    ENSMUSG00000114515\nName: gene_identifier, dtype: object'
'gene_symbol\nAldoa    ENSMUSG00000114515\nName: gene_identifier, dtype: object'
'gene_symbol\nArhgef4    ENSMUSG00000037509\nArhgef4    ENSMUSG00000118272\nName: gene_identifier, dtype: object'
'gene_symbol\nArhgef4    ENSMUSG00000118272\nName: gene_identifier, dtype: object'
'gene_symbol\nBtbd35f1    ENSMUSG00000118409\nName: gene_identifier, dtype: object'
'gene_symbol\nBtbd35f1    ENSMUSG00000118459\nBtbd35f1    ENSMUSG00000118409\nName: gene_identifier, dtype: object'
'gene_symbol\nC730027H18Rik    ENSMUSG00000112189\nName: gene_identifier, dtype: object'
'gene_symbol\nC730027H18Rik    ENSMUSG00000112366\nC730027H18Rik    ENSMU

In [37]:
# Identify positions of unmatched gene identifiers.
mismatch_positions = [
    i for i, gene_id in enumerate(expression_gene_ids)
    if gene_id not in metadata_gene_ids
]

# Compare the first 10 mismatched positions with the metadata.
for i in mismatch_positions[:10]:
    print(f"\nPosition: {i}")
    print("Expression ID:", repr(expression_gene_ids[i]))
    print("Metadata ID at same position:",
          gene_metadata.iloc[i]["gene_identifier"])
    print("Metadata gene symbol:",
          gene_metadata.iloc[i]["gene_symbol"])


Position: 132
Expression ID: 'gene_symbol\nPtp4a1    ENSMUSG00000026064\nPtp4a1    ENSMUSG00000117310\nName: gene_identifier, dtype: object'
Metadata ID at same position: ENSMUSG00000026064
Metadata gene symbol: Ptp4a1

Position: 133
Expression ID: 'gene_symbol\nPtp4a1    ENSMUSG00000117310\nName: gene_identifier, dtype: object'
Metadata ID at same position: ENSMUSG00000117310
Metadata gene symbol: Ptp4a1

Position: 164
Expression ID: 'gene_symbol\nArhgef4    ENSMUSG00000037509\nArhgef4    ENSMUSG00000118272\nName: gene_identifier, dtype: object'
Metadata ID at same position: ENSMUSG00000037509
Metadata gene symbol: Arhgef4

Position: 165
Expression ID: 'gene_symbol\nArhgef4    ENSMUSG00000118272\nName: gene_identifier, dtype: object'
Metadata ID at same position: ENSMUSG00000118272
Metadata gene symbol: Arhgef4

Position: 836
Expression ID: 'gene_symbol\nSept2    ENSMUSG00000026276\nSept2    ENSMUSG00000116048\nName: gene_identifier, dtype: object'
Metadata ID at same position: ENSMU

In [38]:
# Restore gene identifiers from the matching metadata positions.
adata.var_names = gene_metadata["gene_identifier"].astype(str).to_numpy()

# Verify that all gene identifiers are now aligned.
expression_gene_ids = pd.Index(adata.var_names.astype(str))
metadata_gene_ids = pd.Index(gene_metadata["gene_identifier"].astype(str))

print("Matching gene identifiers:",
      len(expression_gene_ids.intersection(metadata_gene_ids)))
print("Unique expression identifiers:", expression_gene_ids.nunique())
print("Total genes:", adata.n_vars)

Matching gene identifiers: 32285
Unique expression identifiers: 32285
Total genes: 32285


In [39]:
# Create a mapping from each donor label to its row index.
donor_to_idx = dict(zip(donor_info["donor_label"], donor_info.index))

# Initialize the donor-level gene count matrix.
# Rows represent donors and columns represent genes.
donor_counts = np.zeros(
    (len(donor_info), adata.n_vars),
    dtype=np.float64
)

# Display the dimensions of the initialized matrix.
print("Donor count matrix shape:", donor_counts.shape)

Donor count matrix shape: (108, 32285)


In [40]:
# Aggregate raw gene counts for each donor in chunks.
for X_chunk, start, end in adata.chunked_X(10_000):

    # Retrieve donor labels for the cells in the current chunk.
    chunk_labels = cell_metadata.iloc[start:end]["donor_label"].to_numpy()

    # Sum gene counts separately for each donor.
    for donor in np.unique(chunk_labels):
        donor_idx = donor_to_idx[donor]
        mask = chunk_labels == donor

        donor_counts[donor_idx, :] += np.asarray(
            X_chunk[mask].sum(axis=0)
        ).ravel()

# Report the final dimensions of the aggregated count matrix.
print("Aggregated donor counts:", donor_counts.shape)
print("Total aggregated counts:", int(donor_counts.sum()))

Aggregated donor counts: (108, 32285)
Total aggregated counts: 24037529679


In [41]:
# Calculate the total number of counts for each donor.
library_size = donor_counts.sum(axis=1)

# Check that every donor has a positive library size.
if np.any(library_size <= 0):
    raise ValueError("At least one donor has zero total counts.")

# Normalize counts to counts per million (CPM) and apply log1p.
log_cpm = np.log1p(
    donor_counts / library_size[:, None] * 1_000_000
)

# Display the dimensions and check for invalid values.
print("log1p(CPM) matrix shape:", log_cpm.shape)
print("Contains only finite values:", np.isfinite(log_cpm).all())

log1p(CPM) matrix shape: (108, 32285)
Contains only finite values: True


In [42]:
# Import the statistical test for comparing two independent groups.
from scipy.stats import ttest_ind

# Define the two age groups to compare.
age1 = "P53"
age2 = "P54"

# Select donors belonging to each age group.
idx1 = donor_info["donor_age"].eq(age1).to_numpy()
idx2 = donor_info["donor_age"].eq(age2).to_numpy()

# Perform Welch's t-test for each gene.
t_stat, p_values = ttest_ind(
    log_cpm[idx1],
    log_cpm[idx2],
    axis=0,
    equal_var=False
)

# Report the number of donors and genes tested.
print(f"{age1} donors:", int(idx1.sum()))
print(f"{age2} donors:", int(idx2.sum()))
print("Genes tested:", len(p_values))
print("Finite p-values:", np.isfinite(p_values).sum())

P53 donors: 2
P54 donors: 4
Genes tested: 32285
Finite p-values: 29173


In [43]:
# Import the Benjamini-Hochberg multiple-testing correction.
from statsmodels.stats.multitest import multipletests

# Replace non-finite p-values with 1.0 for FDR correction.
p_values_clean = np.where(
    np.isfinite(p_values),
    p_values,
    1.0
)

# Adjust p-values to control the false discovery rate.
fdr = multipletests(
    p_values_clean,
    method="fdr_bh"
)[1]

# Report the number of valid FDR values and significant genes.
print("Finite FDR values:", np.isfinite(fdr).sum())
print("Significant genes (FDR < 0.05):", int((fdr < 0.05).sum()))

Finite FDR values: 32285
Significant genes (FDR < 0.05): 0


In [44]:
# Create a results table for the P53 versus P54 comparison.
de_p53_p54 = pd.DataFrame({
    "age_1": age1,
    "age_2": age2,
    "gene_identifier": gene_metadata["gene_identifier"].astype(str).to_numpy(),
    "gene_symbol": gene_metadata["gene_symbol"].to_numpy(),
    "p_value": p_values,
    "FDR": fdr,
})

# Save the differential expression results to a CSV file.
output_path = "/content/DE_P53_vs_P54.csv"
de_p53_p54.to_csv(output_path, index=False)

# Report the output file and table dimensions.
print("Results shape:", de_p53_p54.shape)
print("Saved to:", output_path)

# Display the first five rows.
display(de_p53_p54.head())

Results shape: (32285, 6)
Saved to: /content/DE_P53_vs_P54.csv


,age_1,age_2,gene_identifier,gene_symbol,p_value,FDR
0,P53,P54,ENSMUSG00000051951,Xkr4,0.340706,0.949053
1,P53,P54,ENSMUSG00000089699,Gm1992,0.637817,0.998590
2,P53,P54,ENSMUSG00000102331,Gm19938,0.350075,0.949053
3,P53,P54,ENSMUSG00000102343,Gm37381,0.378850,0.949053
4,P53,P54,ENSMUSG00000025900,Rp1,0.052952,0.949053


In [46]:
# Import tools for generating combinations.
import itertools

# Identify age groups with at least two independent donors.
valid_ages = [
    age
    for age, n in donor_info.groupby("donor_age")["donor_label"].nunique().items()
    if n >= 2
]

# Calculate the number of pairwise age comparisons.
n_comparisons = len(list(itertools.combinations(valid_ages, 2)))

# Display eligible age groups and comparisons.
print("Eligible age groups:", len(valid_ages))
print("Pairwise comparisons:", n_comparisons)
print(
    "Excluded age groups:",
    sorted(set(donor_info["donor_age"]) - set(valid_ages))
)

Eligible age groups: 28
Pairwise comparisons: 378
Excluded age groups: ['P551', 'P57']


In [47]:
# Create the output directory for pairwise differential expression tables.
import os
from pathlib import Path

OUTPUT_DIR = Path("/content/DE_subtables")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Store summary information for every age-pair comparison.
summary_rows = []

# Run differential expression analysis for every eligible age pair.
for age1, age2 in itertools.combinations(valid_ages, 2):

    # Select donors belonging to each age group.
    idx1 = donor_info["donor_age"].eq(age1).to_numpy()
    idx2 = donor_info["donor_age"].eq(age2).to_numpy()

    # Perform Welch's t-test for each gene.
    _, p_values_pair = ttest_ind(
        log_cpm[idx1],
        log_cpm[idx2],
        axis=0,
        equal_var=False
    )

    # Replace non-finite p-values for multiple-testing correction.
    p_values_clean = np.where(
        np.isfinite(p_values_pair),
        p_values_pair,
        1.0
    )

    # Apply Benjamini-Hochberg FDR correction.
    fdr_pair = multipletests(
        p_values_clean,
        method="fdr_bh"
    )[1]

    # Create the results table for this age pair.
    de_table = pd.DataFrame({
        "age_1": age1,
        "age_2": age2,
        "gene_identifier": gene_metadata["gene_identifier"].astype(str).to_numpy(),
        "gene_symbol": gene_metadata["gene_symbol"].to_numpy(),
        "p_value": p_values_pair,
        "FDR": fdr_pair
    })

    # Save the results table as a CSV file.
    safe_age1 = age1.replace(" ", "_")
    safe_age2 = age2.replace(" ", "_")
    filename = f"DE_{safe_age1}_vs_{safe_age2}.csv"
    de_table.to_csv(OUTPUT_DIR / filename, index=False)

    # Record the number of significant genes in this comparison.
    summary_rows.append({
        "comparison": f"{safe_age1}_vs_{safe_age2}",
        "age_1": age1,
        "age_2": age2,
        "genes_tested": len(de_table),
        "significant_genes_FDR_0.05": int((fdr_pair < 0.05).sum())
    })

    # Print progress after each comparison.
    if len(summary_rows) % 25 == 0:
        print(f"Completed {len(summary_rows)} / {n_comparisons} comparisons")

# Create and save the summary table.
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("/content/DE_comparison_summary.csv", index=False)

# Report the final number of completed comparisons and files.
print("Completed comparisons:", len(summary_df))
print("CSV files created:", len(list(OUTPUT_DIR.glob("*.csv"))))
print("Summary saved to: /content/DE_comparison_summary.csv")

Completed 25 / 378 comparisons
Completed 50 / 378 comparisons
Completed 75 / 378 comparisons
Completed 100 / 378 comparisons
Completed 125 / 378 comparisons
Completed 150 / 378 comparisons
Completed 175 / 378 comparisons
Completed 200 / 378 comparisons
Completed 225 / 378 comparisons
Completed 250 / 378 comparisons
Completed 275 / 378 comparisons
Completed 300 / 378 comparisons
Completed 325 / 378 comparisons
Completed 350 / 378 comparisons
Completed 375 / 378 comparisons
Completed comparisons: 378
CSV files created: 378
Summary saved to: /content/DE_comparison_summary.csv


In [48]:
# Count the exported differential expression tables.
csv_files = list(OUTPUT_DIR.glob("*.csv"))

# Create a ZIP archive containing all comparison tables.
import shutil

zip_path = shutil.make_archive(
    "/content/DE_subtables",
    "zip",
    str(OUTPUT_DIR)
)

# Report the number of files and archive size.
print("CSV comparison files:", len(csv_files))
print("Summary table exists:", os.path.exists("/content/DE_comparison_summary.csv"))
print(f"ZIP archive size: {os.path.getsize(zip_path) / (1024**2):.1f} MB")
print("ZIP archive saved to:", zip_path)

CSV comparison files: 378
Summary table exists: True
ZIP archive size: 217.4 MB
ZIP archive saved to: /content/DE_subtables.zip


In [49]:
# Import the Google Colab file download utility.
from google.colab import files

# Download the ZIP archive containing all 378 comparisons.
files.download("/content/DE_subtables.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [50]:
# Download the summary of all 378 age-group comparisons.
from google.colab import files
files.download("/content/DE_comparison_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>